# AI Stock News (Ollama)

This notebook fetches stock news and summarizes the top items using your local Ollama server.

In [7]:
import json
import os
import re
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import pandas as pd
import yfinance as yf
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / '.env')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434').strip().rstrip('/')
if OLLAMA_BASE_URL == 'http://localhost:11434':
    OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', '').strip()
STOCK_QUERY = 'RELIANCE'

def ollama_is_reachable(base_url: str) -> bool:
    if not base_url:
        return False
    try:
        with urlopen(f"{base_url}/api/tags", timeout=5) as response:
            return response.status == 200
    except Exception:
        return False

missing = []
if not OLLAMA_BASE_URL:
    missing.append('OLLAMA_BASE_URL')
if not OLLAMA_MODEL:
    missing.append('OLLAMA_MODEL')

if missing:
    print('Missing in .env:', ', '.join(missing))
elif not ollama_is_reachable(OLLAMA_BASE_URL):
    print(f'Ollama server is not reachable at {OLLAMA_BASE_URL}')
    print('Start it first, then rerun this cell and Cell 4: ollama serve')
else:
    print('Ollama config loaded from .env and server is reachable')

Ollama config loaded from .env and server is reachable


In [5]:
def normalize_ticker(stock_query: str) -> str:
    ticker = stock_query.strip().upper()
    if not ticker:
        return ticker
    if ticker.startswith('^') or ticker.endswith('.NS') or ticker.endswith('.BO'):
        return ticker
    return f'{ticker}.NS'

def fetch_stock_news(stock_query: str, limit: int = 50):
    ticker = normalize_ticker(stock_query)
    candidates = [ticker, stock_query.strip().upper(), f'{stock_query.strip().upper()}.BO']
    raw_news = []

    for candidate in candidates:
        if not candidate:
            continue
        try:
            items = yf.Ticker(candidate).news or []
            if items:
                raw_news = items
                ticker = candidate
                break
        except Exception:
            continue

    return ticker, raw_news[:limit]

def extract_fields(items):
    cleaned = []
    for item in items:
        content = item.get('content', {}) if isinstance(item, dict) else {}
        provider = content.get('provider', {}) if isinstance(content, dict) else {}
        canonical_url = content.get('canonicalUrl', {}) if isinstance(content, dict) else {}
        click_url = content.get('clickThroughUrl', {}) if isinstance(content, dict) else {}

        raw_pub = content.get('pubDate') or item.get('providerPublishTime')
        pub_ts = pd.to_datetime(raw_pub, errors='coerce', utc=True)
        pub_text = pub_ts.strftime('%Y-%m-%d %H:%M') if pd.notna(pub_ts) else 'N/A'

        cleaned.append({
            'title': content.get('title') or item.get('title', 'Untitled'),
            'source': provider.get('displayName') or content.get('provider') or item.get('publisher', 'Unknown'),
            'url': canonical_url.get('url') or click_url.get('url') or item.get('link', ''),
            'published_at': pub_text,
        })

    return (
        pd.DataFrame(cleaned)
        .sort_values('published_at', ascending=False, na_position='last')
        .reset_index(drop=True)
        .to_dict('records') if cleaned else []
    )

def parse_json_response(text: str):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\\s*```$', '', cleaned)
    return json.loads(cleaned)

ticker, raw_news = fetch_stock_news(STOCK_QUERY)
news_items = extract_fields(raw_news)

latest_news_df = pd.DataFrame(news_items)[['published_at', 'source', 'title', 'url']].head(10) if news_items else pd.DataFrame()

print(f'News source ticker: {ticker}')
print(f'Total items fetched: {len(news_items)}')
print('Showing latest 10 headlines (newest first):')
latest_news_df

News source ticker: RELIANCE.NS
Total items fetched: 10
Showing latest 10 headlines (newest first):


,published_at,source,title,url
0,2026-04-03 14:07,Simply Wall St.,How The Investment Story For Reliance Industri...,https://finance.yahoo.com/markets/stocks/artic...
1,2026-04-01 01:51,Reuters,India diesel exports to SE Asia hit 7-year hig...,https://sg.finance.yahoo.com/news/india-diesel...
2,2026-03-26 12:28,GuruFocus.com,OpenAI Appoints JioStar CEO to Lead Asia Expan...,https://finance.yahoo.com/sectors/technology/a...
3,2026-03-24 10:06,Reuters,Exclusive-India's Reliance buys 5 million barr...,https://sg.finance.yahoo.com/news/exclusive-in...
4,2026-03-23 09:38,Reuters,Factbox-Ambani's Reliance Jio: businesses and ...,https://sg.finance.yahoo.com/news/factbox-amba...
5,2026-03-23 04:53,Reuters,India Coca‑Cola bottler SLMG says Middle East ...,https://sg.finance.yahoo.com/news/india-coca-c...
6,2026-03-20 10:06,Simply Wall St.,How The Reliance Industries (NSEI:RELIANCE) St...,https://finance.yahoo.com/markets/stocks/artic...
7,2026-03-19 08:46,Reuters,"Factbox-India, a major energy consumer and ref...",https://sg.finance.yahoo.com/news/factbox-indi...
8,2026-03-18 19:52,Reuters,"India orders oil, gas firms to share import, e...",https://sg.finance.yahoo.com/news/india-orders...
9,2026-03-18 10:37,ESG Dive,"Reliance Industries, Samsung subsidiary sign $...",https://www.esgdive.com/news/reliance-industri...


In [6]:
def call_ollama(prompt: str) -> str:
    endpoint = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        'model': OLLAMA_MODEL,
        'prompt': prompt,
        'format': 'json',
        'stream': False,
        'options': {'temperature': 0.1},
    }

    request = Request(
        endpoint,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )

    with urlopen(request, timeout=120) as response:
        body = response.read().decode('utf-8')
    return json.loads(body).get('response', '').strip()

if not news_items:
    print('No news found. Try another stock symbol.')
elif not OLLAMA_BASE_URL or not OLLAMA_MODEL:
    print('Set OLLAMA_BASE_URL and OLLAMA_MODEL in .env, then rerun Cell 2.')
elif not ollama_is_reachable(OLLAMA_BASE_URL):
    print(f'Cannot connect to Ollama at {OLLAMA_BASE_URL}')
    print('Run this in terminal: ollama serve')
    print(f'And ensure model exists: ollama pull {OLLAMA_MODEL}')
else:
    prompt = f'''
You are a market news analyst.
Pick the 10 most important headlines and summarize each briefly for investors.
Use only the provided items.
Return valid JSON only in this exact structure:
{{"summary": string, "highlights": [{{"title": string, "source": string, "url": string, "published_at": string, "summary": string}}]}}

Stock: {ticker}
News items: {json.dumps(news_items, ensure_ascii=False, indent=2)}
'''

    try:
        text = call_ollama(prompt)
        payload = parse_json_response(text)

        print('Summary:')
        print(payload.get('summary', ''))
        print('\nTop highlights:')
        pd.DataFrame(payload.get('highlights', [])).head(10)
    except URLError as exc:
        print('Ollama connection error:', exc)
        print('Start Ollama: ollama serve')
    except Exception as exc:
        print('Ollama call or JSON parsing failed:')
        print(exc)

Summary:
Reliance Industries (NSEI:RELIANCE) is currently undergoing a significant shift in its business strategy, with a focus on increasing its presence in the Indian market and potentially exploring new areas like renewable energy. The company is also undergoing a significant restructuring to improve its financial stability and operational efficiency.

Top highlights:
